<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake/Multi_Agent_Physics_with_Controllers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class Controller:
    """
    Base class for all controllers.
    """

    def compute_control(self, t, self_state, neighbor_states):
        """
        Compute force and torque for one agent.

        Parameters
        ----------
        t : float
            Current simulation time.

        self_state : AgentState
            Current state of this agent.

        neighbor_states : dict
            Neighbor agent states. Empty in Milestone 3.

        Returns
        -------
        force : np.ndarray, shape (3,)
            Force expressed in world frame.

        torque : np.ndarray, shape (3,)
            Torque expressed in world frame.
        """
        raise NotImplementedError

In [ ]:
class ZeroController(Controller):
    """
    Controller that applies no force and no torque.
    Useful as a default controller.
    """

    def compute_control(self, t, self_state, neighbor_states):
        force = np.zeros(3)
        torque = np.zeros(3)
        return force, torque

In [ ]:
class Agent:
    def __init__(self, agent_id, config, controller=None):
        self.id = agent_id
        self.config = config
        self.controller = (
            controller
            if controller is not None
            else ZeroController()
        )

        self.body = None

In [ ]:
class AgentState:
    """
    Clean simulator-level state of one rigid-body agent.

    This class hides Drake-specific pose and velocity objects.
    Controllers and loggers should use this class instead of directly
    using Drake's q, v, RigidTransform, or SpatialVelocity objects.
    """

    def __init__(self,
                 position,
                 orientation,
                 linear_velocity,
                 angular_velocity):

        self.position = np.array(position, dtype=float)
        self.orientation = np.array(orientation, dtype=float)
        self.linear_velocity = np.array(linear_velocity, dtype=float)
        self.angular_velocity = np.array(angular_velocity, dtype=float)

        self._validate()

    def _validate(self):
        if self.position.shape != (3,):
            raise ValueError("Position must be a 3D vector.")

        if self.orientation.shape != (3, 3):
            raise ValueError("Orientation must be a 3x3 rotation matrix.")

        if not np.allclose(self.orientation.T @ self.orientation, np.eye(3), atol=1e-6):
            raise ValueError("Orientation must satisfy R^T R = I.")

        if not np.isclose(np.linalg.det(self.orientation), 1.0, atol=1e-6):
            raise ValueError("Orientation matrix must have determinant +1.")

        if self.linear_velocity.shape != (3,):
            raise ValueError("Linear velocity must be a 3D vector.")

        if self.angular_velocity.shape != (3,):
            raise ValueError("Angular velocity must be a 3D vector.")

    def as_dict(self):
        return {
            "position": self.position,
            "orientation": self.orientation,
            "linear_velocity": self.linear_velocity,
            "angular_velocity": self.angular_velocity
        }

In [ ]:
import numpy as np
class EnvironmentConfig:
    def __init__(self, gravity=None):
        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else np.array([0.0, 0.0, -9.81])
        )

        self._validate()

    def _validate(self):
        if self.gravity.shape != (3,):
            raise ValueError("Gravity must be a 3D vector.")

In [ ]:
import numpy as np

class BodyConfig:
    """
    Configuration for a general rigid body.
    Defined purely by mass and inertia tensor.
    """

    def __init__(self,
                 mass: float,
                 inertia_matrix: np.ndarray,
                 initial_position=None,
                 initial_orientation=None,
                 initial_angular_velocity=None,
                 initial_linear_velocity=None,
                 #gravity=None,
                 friction=(0.9, 0.8),
                 size=(0.2, 0.2, 0.2),
                 color=(0.2, 0.6, 1.0, 1.0)
                 ):

        # ---- Fundamental physical properties ----
        self.mass = float(mass)
        self.inertia_matrix = np.array(inertia_matrix, dtype=float)

        # ---- Initial position ----
        self.initial_position = (
            np.array(initial_position, dtype=float)
            if initial_position is not None
            else np.array([0.0, 0.0, 1.0])
        )

        # Orientation stored as rotation matrix (3x3)
        self.initial_orientation = (
            np.array(initial_orientation, dtype=float)
            if initial_orientation is not None
            else np.eye(3)
        )

        # ---- Initial Angular velocity ----
        self.initial_angular_velocity = (
            np.array(initial_angular_velocity, dtype=float)
            if initial_angular_velocity is not None
            else np.zeros(3)
        )

        # ---- Initial Linear velocity ----
        self.initial_linear_velocity = (
            np.array(initial_linear_velocity, dtype=float)
            if initial_linear_velocity is not None
            else np.zeros(3)
        )

        #---- Optional simulation properties ----
        '''
        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else np.array([0.0, 0.0, -9.81])
        )
        '''

        self.friction = friction
        self.size = np.array(size, dtype=float)
        self.color = np.array(color, dtype=float)

        # ---- Validate physical correctness ----
        self._validate()

    def _validate(self):
        if self.mass <= 0:
            raise ValueError("Mass must be positive.")

        if self.inertia_matrix.shape != (3, 3):
            raise ValueError("Inertia matrix must be 3x3.")

        # Must be symmetric
        if not np.allclose(self.inertia_matrix,
                           self.inertia_matrix.T):
            raise ValueError("Inertia matrix must be symmetric.")

        # Must be positive definite
        eigenvalues = np.linalg.eigvalsh(self.inertia_matrix)
        if not np.all(eigenvalues > 0):
            raise ValueError(
                "Inertia matrix must be positive definite."
            )

        # ---- Rotation matrix validity ----
        R = self.initial_orientation

        if R.shape != (3, 3):
            raise ValueError("Initial orientation must be 3x3.")

        # Check orthogonality: R^T R ≈ I
        if not np.allclose(R.T @ R, np.eye(3), atol=1e-6):
            raise ValueError("Orientation matrix must be orthogonal (R^T R = I).")

        # Check determinant: det(R) ≈ 1
        det_R = np.linalg.det(R)
        if not np.isclose(det_R, 1.0, atol=1e-6):
            raise ValueError("Orientation matrix must have determinant +1.")

        # ---- Initial Angular and Linear Velocities validity ----
        if self.initial_linear_velocity.shape != (3,):
            raise ValueError("Initial linear velocity must be a 3D vector.")

        if self.initial_angular_velocity.shape != (3,):
            raise ValueError("Initial angular velocity must be a 3D vector.")

        if self.size.shape != (3,):
            raise ValueError("Body size must be a 3D vector.")

        if np.any(self.size <= 0):
            raise ValueError("Body size values must be positive.")

        if self.color.shape != (4,):
            raise ValueError("Color must be RGBA with 4 values.")



In [ ]:
from pydrake.all import LeafSystem, Value, ExternallyAppliedSpatialForce, SpatialForce
import numpy as np


class ExternalForceSystem(LeafSystem):
    """
    Applies each agent's controller-generated force and torque to Drake.
    """

    def __init__(self, plant, agents):
        super().__init__()

        self.plant = plant
        self.agents = agents

        self.DeclareVectorInputPort(
            "plant_state",
            plant.num_positions() + plant.num_velocities()
        )

        self.DeclareAbstractOutputPort(
            "spatial_forces",
            lambda: Value([ExternallyAppliedSpatialForce()]),
            self.CalcOutput
        )

    def CalcOutput(self, context, output):
        forces = []

        # Current simulation time
        t = context.get_time()

        # Read full Drake plant state
        state = self.EvalVectorInput(context, 0).value()

        # Create a temporary plant context for state extraction
        plant_context = self.plant.CreateDefaultContext()
        self.plant.SetPositionsAndVelocities(
            plant_context,
            state
        )

        # Convert Drake state to AgentState objects
        all_states = {}

        for agent in self.agents:
            pose = self.plant.GetFreeBodyPose(
                plant_context,
                agent.body
            )

            velocity = self.plant.EvalBodySpatialVelocityInWorld(
                plant_context,
                agent.body
            )

            all_states[agent.id] = AgentState(
                position=pose.translation(),
                orientation=pose.rotation().matrix(),
                linear_velocity=velocity.translational(),
                angular_velocity=velocity.rotational()
            )

        # Ask each agent controller for force and torque
        for agent in self.agents:
            self_state = all_states[agent.id]

            # No neighbor communication yet in Milestone 4
            neighbor_states = {}

            force_vec, torque_vec = agent.controller.compute_control(
                t=t,
                self_state=self_state,
                neighbor_states=neighbor_states
            )

            force_vec = np.array(force_vec, dtype=float)
            torque_vec = np.array(torque_vec, dtype=float)

            if force_vec.shape != (3,):
                raise ValueError(
                    f"Controller force for Agent {agent.id} must be a 3D vector."
                )

            if torque_vec.shape != (3,):
                raise ValueError(
                    f"Controller torque for Agent {agent.id} must be a 3D vector."
                )

            spatial_force = ExternallyAppliedSpatialForce()
            spatial_force.body_index = agent.body.index()

            # Apply force at body origin/COM
            spatial_force.p_BoBq_B = np.zeros(3)

            spatial_force.F_Bq_W = SpatialForce(
                tau=torque_vec,
                f=force_vec
            )

            forces.append(spatial_force)

        output.set_value(forces)

In [ ]:
from pydrake.all import *
import numpy as np
from pydrake.all import SpatialVelocity

class MultiAgentSimulator:
    """
    Drake-based simulator for multiple free rigid-body agents
    with configurable environment and external spatial forces.
    """
    def __init__(self, agents, time_step: float = 0.001, environment=None):
        self.agents = agents
        self.agent_map = {agent.id: agent for agent in agents}
        self.time_step = time_step
        self.environment = environment or EnvironmentConfig()

        self._build_system()

    def _build_system(self):
        # 1. Diagram & plant
        self.builder = DiagramBuilder()
        self.plant, self.scene_graph = AddMultibodyPlantSceneGraph(
            self.builder,
            MultibodyPlant(time_step=self.time_step)
        )

        # 2. Add agents
        for agent in self.agents:
            self._add_agent_body(agent)

        # 3. Add ground
        #self._add_ground()

        # Gravity
        self.plant.mutable_gravity_field().set_gravity_vector(self.environment.gravity)

        # 4. Finalize plant
        self.plant.Finalize()

        # 4.1 Add external force system if provided
        # 4.1 Add controller-based external force system
        self.force_system = ExternalForceSystem(
            self.plant,
            self.agents
        )

        self.builder.AddSystem(self.force_system)

        self.builder.Connect(
            self.plant.get_state_output_port(),
            self.force_system.get_input_port(0)
        )

        self.builder.Connect(
            self.force_system.get_output_port(),
            self.plant.get_applied_spatial_force_input_port()
        )


        # 5. Meshcat visualizer (optional, for debugging)
        self.meshcat = StartMeshcat()
        MeshcatVisualizer.AddToBuilder(
            self.builder,
            self.scene_graph,
            self.meshcat
        )

        # 6. Build diagram
        self.diagram = self.builder.Build()
        self.simulator = Simulator(self.diagram)

        # ---- SET INITIAL VELOCITIES ----
        context = self.simulator.get_mutable_context()
        plant_context = self.plant.GetMyContextFromRoot(context)

        for agent in self.agents:

            cfg = agent.config

            V_WB = SpatialVelocity(
                w=cfg.initial_angular_velocity,
                v=cfg.initial_linear_velocity
            )

            self.plant.SetFreeBodySpatialVelocity(
                agent.body,
                V_WB,
                plant_context
            )

    def _add_agent_body(self, agent):
        cfg = agent.config

        # Convert inertia matrix to UnitInertia
        I = cfg.inertia_matrix
        m = cfg.mass

        unit_inertia = UnitInertia(
            Ixx=I[0, 0]/m,
            Iyy=I[1, 1]/m,
            Izz=I[2, 2]/m,
            Ixy=I[0, 1]/m,
            Ixz=I[0, 2]/m,
            Iyz=I[1, 2]/m
        )

        spatial_inertia = SpatialInertia(
            mass=cfg.mass,
            p_PScm_E=np.zeros(3),
            G_SP_E=unit_inertia
        )

        body = self.plant.AddRigidBody(
            f"body_{agent.id}",
            spatial_inertia
        )
        agent.body = body

        # Set initial pose
        X_WB = RigidTransform(
            RotationMatrix(cfg.initial_orientation),
            cfg.initial_position
        )

        self.plant.SetDefaultFloatingBaseBodyPose(
            agent.body,
            X_WB
        )

        # Minimal placeholder collision geometry
        box_size = cfg.size
        collision_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterCollisionGeometry(
            agent.body,
            RigidTransform(),
            collision_shape,
            "body_collision",
            CoulombFriction(*cfg.friction)
        )

        # Minimal visual geometry (for Meshcat visualization)
        visual_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterVisualGeometry(
            agent.body,
            RigidTransform(),
            visual_shape,
            "body_visual",
            cfg.color  # RGBA color
        )

    def get_agent_state(self, agent, plant_context=None):
        """
        Extract the current physical state of one agent from Drake
        and convert it into an AgentState object.
        """

        if plant_context is None:
            context = self.simulator.get_context()
            plant_context = self.plant.GetMyContextFromRoot(context)

        pose = self.plant.GetFreeBodyPose(
            plant_context,
            agent.body
        )

        velocity = self.plant.EvalBodySpatialVelocityInWorld(
            plant_context,
            agent.body
        )

        return AgentState(
            position=pose.translation(),
            orientation=pose.rotation().matrix(),
            linear_velocity=velocity.translational(),
            angular_velocity=velocity.rotational()
        )

    def _add_ground(self):
        ground_shape = HalfSpace()
        X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0), [0, 0, 0])

        self.plant.RegisterCollisionGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_collision",
            CoulombFriction(0.9, 0.8)
        )

        self.plant.RegisterVisualGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_visual",
            [0.5, 0.5, 0.5, 1.0]
        )

    def simulate(self, duration: float = 5.0, realtime_rate: float = 1.0):
        self.simulator.set_target_realtime_rate(realtime_rate)
        self.simulator.Initialize()
        self.simulator.AdvanceTo(duration)

    def get_web_url(self) -> str:
        return self.meshcat.web_url()

    def get_all_agent_states(self):
        """
        Return all agent states as AgentState objects.

        This version is intended for controllers, loggers, and metrics.
        """

        context = self.simulator.get_context()
        plant_context = self.plant.GetMyContextFromRoot(context)

        states = {}

        for agent in self.agents:
            states[agent.id] = self.get_agent_state(agent, plant_context)

        return states

In [ ]:
class ConstantForceController(Controller):
    """
    Applies a constant force and torque to an agent.
    """

    def __init__(self, force=None, torque=None):
        self.force = (
            np.array(force, dtype=float)
            if force is not None
            else np.zeros(3)
        )

        self.torque = (
            np.array(torque, dtype=float)
            if torque is not None
            else np.zeros(3)
        )

        self._validate()

    def _validate(self):
        if self.force.shape != (3,):
            raise ValueError("Force must be a 3D vector.")

        if self.torque.shape != (3,):
            raise ValueError("Torque must be a 3D vector.")

    def compute_control(self, t, self_state, neighbor_states):
        return self.force, self.torque


class PositionPDController(Controller):
    """
    Position PD controller with optional gravity compensation.

    This controller moves the agent toward a desired position using
    position and velocity feedback.

    Force is expressed in the world frame.
    Torque is zero in this controller.
    """

    def __init__(self,
                 desired_position,
                 desired_velocity=None,
                 kp=2.0,
                 kd=2.0,
                 mass=None,
                 gravity=None,
                 force_limit=None):

        self.desired_position = np.array(desired_position, dtype=float)

        self.desired_velocity = (
            np.array(desired_velocity, dtype=float)
            if desired_velocity is not None
            else np.zeros(3)
        )

        self.kp = float(kp)
        self.kd = float(kd)

        self.mass = (
            float(mass)
            if mass is not None
            else None
        )

        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else None
        )

        self.force_limit = (
            float(force_limit)
            if force_limit is not None
            else None
        )

        self._validate()

    def _validate(self):
        if self.desired_position.shape != (3,):
            raise ValueError("Desired position must be a 3D vector.")

        if self.desired_velocity.shape != (3,):
            raise ValueError("Desired velocity must be a 3D vector.")

        if self.kp < 0:
            raise ValueError("kp must be non-negative.")

        if self.kd < 0:
            raise ValueError("kd must be non-negative.")

        if self.mass is not None and self.mass <= 0:
            raise ValueError("Mass must be positive.")

        if self.gravity is not None and self.gravity.shape != (3,):
            raise ValueError("Gravity must be a 3D vector.")

        if self.gravity is not None and self.mass is None:
            raise ValueError("Mass must be provided when gravity compensation is used.")

        if self.force_limit is not None and self.force_limit <= 0:
            raise ValueError("force_limit must be positive.")

    def compute_control(self, t, self_state, neighbor_states):
        position_error = self.desired_position - self_state.position
        velocity_error = self.desired_velocity - self_state.linear_velocity

        # PD feedback force
        force = self.kp * position_error + self.kd * velocity_error

        # Gravity compensation
        if self.gravity is not None:
            force = force - self.mass * self.gravity

        # Optional safety limit
        if self.force_limit is not None:
            norm_force = np.linalg.norm(force)

            if norm_force > self.force_limit:
                force = force / norm_force * self.force_limit

        torque = np.zeros(3)

        return force, torque

def vee(skew_matrix):
    """
    Convert a 3x3 skew-symmetric matrix into a 3D vector.

    For a matrix:
        [  0  -z   y ]
        [  z   0  -x ]
        [ -y   x   0 ]

    vee(matrix) returns:
        [x, y, z]
    """
    return np.array([
        skew_matrix[2, 1],
        skew_matrix[0, 2],
        skew_matrix[1, 0]
    ])


class AttitudePDController(Controller):
    """
    Attitude PD controller for one rigid-body agent.

    This controller rotates the body toward a desired orientation.

    It uses:
        - orientation feedback
        - angular velocity damping

    Force is zero.
    Torque is expressed in the world frame before being returned.
    """

    def __init__(self,
                 desired_orientation=None,
                 kp=1.0,
                 kd=1.0,
                 torque_limit=None):

        self.desired_orientation = (
            np.array(desired_orientation, dtype=float)
            if desired_orientation is not None
            else np.eye(3)
        )

        self.kp = float(kp)
        self.kd = float(kd)

        self.torque_limit = (
            float(torque_limit)
            if torque_limit is not None
            else None
        )

        self._validate()

    def _validate(self):
        R = self.desired_orientation

        if R.shape != (3, 3):
            raise ValueError("Desired orientation must be a 3x3 matrix.")

        if not np.allclose(R.T @ R, np.eye(3), atol=1e-6):
            raise ValueError("Desired orientation must satisfy R^T R = I.")

        if not np.isclose(np.linalg.det(R), 1.0, atol=1e-6):
            raise ValueError("Desired orientation must have determinant +1.")

        if self.kp < 0:
            raise ValueError("kp must be non-negative.")

        if self.kd < 0:
            raise ValueError("kd must be non-negative.")

        if self.torque_limit is not None and self.torque_limit <= 0:
            raise ValueError("torque_limit must be positive.")

    def compute_control(self, t, self_state, neighbor_states):
        # Current orientation: body frame B relative to world frame W
        R = self_state.orientation

        # Desired orientation
        R_des = self.desired_orientation

        # Orientation error on SO(3), expressed in the body frame
        error_matrix = 0.5 * (R_des.T @ R - R.T @ R_des)
        orientation_error_body = vee(error_matrix)

        # Current angular velocity is taken from Drake in world frame.
        # Convert it into body frame for the attitude control law.
        angular_velocity_body = R.T @ self_state.angular_velocity

        # PD torque in body frame
        torque_body = (
            -self.kp * orientation_error_body
            -self.kd * angular_velocity_body
        )

        # Convert torque back to world frame because Drake SpatialForce uses world frame here
        torque_world = R @ torque_body

        # Optional torque saturation
        if self.torque_limit is not None:
            norm_torque = np.linalg.norm(torque_world)

            if norm_torque > self.torque_limit:
                torque_world = torque_world / norm_torque * self.torque_limit

        force = np.zeros(3)

        return force, torque_world

In [ ]:
class FullPosePDController(Controller):
    """
    Full pose PD controller for one rigid-body agent.

    This controller controls both:
        - position using force
        - orientation using torque

    Force and torque are returned in the world frame.
    """

    def __init__(self,
                 desired_position,
                 desired_orientation=None,
                 desired_velocity=None,
                 desired_angular_velocity_body=None,
                 kp_pos=4.0,
                 kd_pos=4.0,
                 kp_att=1.0,
                 kd_att=1.0,
                 mass=None,
                 gravity=None,
                 force_limit=None,
                 torque_limit=None):

        # Desired translation
        self.desired_position = np.array(desired_position, dtype=float)

        self.desired_velocity = (
            np.array(desired_velocity, dtype=float)
            if desired_velocity is not None
            else np.zeros(3)
        )

        # Desired orientation
        self.desired_orientation = (
            np.array(desired_orientation, dtype=float)
            if desired_orientation is not None
            else np.eye(3)
        )

        # Desired angular velocity, expressed in body frame
        self.desired_angular_velocity_body = (
            np.array(desired_angular_velocity_body, dtype=float)
            if desired_angular_velocity_body is not None
            else np.zeros(3)
        )

        # Position gains
        self.kp_pos = float(kp_pos)
        self.kd_pos = float(kd_pos)

        # Attitude gains
        self.kp_att = float(kp_att)
        self.kd_att = float(kd_att)

        # Physical parameters for gravity compensation
        self.mass = (
            float(mass)
            if mass is not None
            else None
        )

        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else None
        )

        # Optional safety limits
        self.force_limit = (
            float(force_limit)
            if force_limit is not None
            else None
        )

        self.torque_limit = (
            float(torque_limit)
            if torque_limit is not None
            else None
        )

        self._validate()

    def _validate(self):
        if self.desired_position.shape != (3,):
            raise ValueError("Desired position must be a 3D vector.")

        if self.desired_velocity.shape != (3,):
            raise ValueError("Desired velocity must be a 3D vector.")

        if self.desired_orientation.shape != (3, 3):
            raise ValueError("Desired orientation must be a 3x3 matrix.")

        R = self.desired_orientation

        if not np.allclose(R.T @ R, np.eye(3), atol=1e-6):
            raise ValueError("Desired orientation must satisfy R^T R = I.")

        if not np.isclose(np.linalg.det(R), 1.0, atol=1e-6):
            raise ValueError("Desired orientation must have determinant +1.")

        if self.desired_angular_velocity_body.shape != (3,):
            raise ValueError("Desired angular velocity must be a 3D vector.")

        if self.kp_pos < 0:
            raise ValueError("kp_pos must be non-negative.")

        if self.kd_pos < 0:
            raise ValueError("kd_pos must be non-negative.")

        if self.kp_att < 0:
            raise ValueError("kp_att must be non-negative.")

        if self.kd_att < 0:
            raise ValueError("kd_att must be non-negative.")

        if self.mass is not None and self.mass <= 0:
            raise ValueError("Mass must be positive.")

        if self.gravity is not None and self.gravity.shape != (3,):
            raise ValueError("Gravity must be a 3D vector.")

        if self.gravity is not None and self.mass is None:
            raise ValueError("Mass must be provided when gravity compensation is used.")

        if self.force_limit is not None and self.force_limit <= 0:
            raise ValueError("force_limit must be positive.")

        if self.torque_limit is not None and self.torque_limit <= 0:
            raise ValueError("torque_limit must be positive.")

    def compute_control(self, t, self_state, neighbor_states):
        # -----------------------------
        # Position control: force
        # -----------------------------
        position_error = self.desired_position - self_state.position
        velocity_error = self.desired_velocity - self_state.linear_velocity

        force_world = (
            self.kp_pos * position_error
            + self.kd_pos * velocity_error
        )

        # Gravity compensation
        if self.gravity is not None:
            force_world = force_world - self.mass * self.gravity

        # Optional force saturation
        if self.force_limit is not None:
            norm_force = np.linalg.norm(force_world)

            if norm_force > self.force_limit:
                force_world = force_world / norm_force * self.force_limit

        # -----------------------------
        # Attitude control: torque
        # -----------------------------
        R = self_state.orientation
        R_des = self.desired_orientation

        # Orientation error on SO(3), expressed in body frame
        error_matrix = 0.5 * (R_des.T @ R - R.T @ R_des)
        orientation_error_body = vee(error_matrix)

        # Drake angular velocity is expressed in world frame.
        # Convert it to body frame.
        angular_velocity_body = R.T @ self_state.angular_velocity

        angular_velocity_error_body = (
            angular_velocity_body
            - self.desired_angular_velocity_body
        )

        # Torque in body frame
        torque_body = (
            -self.kp_att * orientation_error_body
            -self.kd_att * angular_velocity_error_body
        )

        # Convert torque to world frame for Drake spatial force
        torque_world = R @ torque_body

        # Optional torque saturation
        if self.torque_limit is not None:
            norm_torque = np.linalg.norm(torque_world)

            if norm_torque > self.torque_limit:
                torque_world = torque_world / norm_torque * self.torque_limit

        return force_world, torque_world

Testing

In [ ]:
import numpy as np

# -----------------------------
# Create body configurations
# -----------------------------

cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.01, 0.01, 0.01]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(1.0, 0.2, 0.2, 1.0)
)

cfg2 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([1, 1, 1]),
    initial_position=np.array([1.0, 0.0, 3.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(0.2, 1.0, 0.2, 1.0)
)

cfg3 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([1, 1, 1]),
    initial_position=np.array([2.0, 0.0, 4.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(0.2, 0.2, 1.0, 1.0)
)

controller1 = ConstantForceController(force=[1, 0, 0])
controller2 = ConstantForceController(
    force=[0, 0, 0],
    torque=[0, 0, 1]
)
controller3 = ConstantForceController(
    force=[0, 0, 0],
    torque=[0, 1, 0]
)
# -----------------------------
# Create agents
# -----------------------------

agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1),
    Agent(agent_id=1, config=cfg2, controller=controller2),
    Agent(agent_id=2, config=cfg3, controller=controller3)
]

# -----------------------------
# Create environment
# -----------------------------

env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, -9.81])
)

# -----------------------------
# Create simulator
# -----------------------------

sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)

In [ ]:
import numpy as np

# -----------------------------
# Environment with normal gravity
# -----------------------------
env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, -9.81])
)

# -----------------------------
# Body configuration
# -----------------------------
cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.01, 0.01, 0.01]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(1.0, 0.2, 0.2, 1.0)
)

# -----------------------------
# Position controller with gravity compensation
# -----------------------------
controller1 = PositionPDController(
    desired_position=[2.0, 0.0, 2.0],
    kp=4.0,
    kd=4.0,
    mass=cfg1.mass,
    gravity=env.gravity,
    force_limit=30.0
)

# -----------------------------
# Agent
# -----------------------------
agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1)
]

# -----------------------------
# Simulator
# -----------------------------
sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)




In [ ]:
import numpy as np

env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, -9.81])
)

cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.01, 0.01, 0.01]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(1.0, 0.2, 0.2, 1.0)
)

cfg2 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.01, 0.01, 0.01]),
    initial_position=np.array([1.0, 0.0, 3.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(0.2, 1.0, 0.2, 1.0)
)

cfg3 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.01, 0.01, 0.01]),
    initial_position=np.array([2.0, 0.0, 4.0]),
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.3, 0.3, 0.3),
    color=(0.2, 0.2, 1.0, 1.0)
)

controller1 = PositionPDController(
    desired_position=[2.0, 0.0, 2.0],
    kp=4.0,
    kd=4.0,
    mass=cfg1.mass,
    gravity=env.gravity,
    force_limit=30.0
)

controller2 = PositionPDController(
    desired_position=[2.0, 1.0, 3.0],
    kp=4.0,
    kd=4.0,
    mass=cfg2.mass,
    gravity=env.gravity,
    force_limit=30.0
)

controller3 = PositionPDController(
    desired_position=[2.0, -1.0, 4.0],
    kp=4.0,
    kd=4.0,
    mass=cfg3.mass,
    gravity=env.gravity,
    force_limit=30.0
)

agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1),
    Agent(agent_id=1, config=cfg2, controller=controller2),
    Agent(agent_id=2, config=cfg3, controller=controller3)
]

sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)



In [ ]:
import numpy as np
from pydrake.all import RollPitchYaw

# -----------------------------
# Initial orientation
# -----------------------------
# Start with the body rotated 45 degrees about z-axis
initial_R = RollPitchYaw(0.0, 0.0, np.pi / 4).ToRotationMatrix().matrix()

# Desired orientation is identity
desired_R = np.eye(3)

# -----------------------------
# Environment
# -----------------------------
env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, 0.0])
)

# -----------------------------
# Body configuration
# -----------------------------
cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.1, 0.1, 0.1]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_orientation=initial_R,
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    initial_angular_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.6, 0.2, 0.2),   # rectangular body, easier to see rotation
    color=(1.0, 0.2, 0.2, 1.0)
)

# -----------------------------
# Attitude controller
# -----------------------------
controller1 = AttitudePDController(
    desired_orientation=desired_R,
    kp=1.0,
    kd=1.0,
    torque_limit=0.5
)

# -----------------------------
# Agent
# -----------------------------
agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1)
]

# -----------------------------
# Simulator
# -----------------------------
sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)


In [ ]:
import numpy as np
from pydrake.all import RollPitchYaw

# -----------------------------
# Desired and initial orientations
# -----------------------------

initial_R = RollPitchYaw(0.0, 0.0, np.pi / 4).ToRotationMatrix().matrix()
desired_R = np.eye(3)

# -----------------------------
# Environment
# -----------------------------

env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, -9.81])
)

# -----------------------------
# Body configuration
# -----------------------------

cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.1, 0.1, 0.1]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_orientation=initial_R,
    initial_linear_velocity=np.array([0.0, 0.0, 0.0]),
    initial_angular_velocity=np.array([0.0, 0.0, 0.0]),
    size=(0.6, 0.2, 0.2),
    color=(1.0, 0.2, 0.2, 1.0)
)

# -----------------------------
# Full pose controller
# -----------------------------

controller1 = FullPosePDController(
    desired_position=[2.0, 0.0, 2.0],
    desired_orientation=desired_R,
    kp_pos=4.0,
    kd_pos=4.0,
    kp_att=1.0,
    kd_att=1.0,
    mass=cfg1.mass,
    gravity=env.gravity,
    force_limit=30.0,
    torque_limit=0.5
)

# -----------------------------
# Agent
# -----------------------------

agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1)
]

# -----------------------------
# Simulator
# -----------------------------

sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)


In [ ]:
import numpy as np
from pydrake.all import RollPitchYaw

env = EnvironmentConfig(
    gravity=np.array([0.0, 0.0, -9.81])
)

R0 = RollPitchYaw(0.0, 0.0, np.pi / 4).ToRotationMatrix().matrix()
R1 = RollPitchYaw(0.0, np.pi / 6, 0.0).ToRotationMatrix().matrix()
R2 = RollPitchYaw(np.pi / 6, 0.0, 0.0).ToRotationMatrix().matrix()

desired_R = np.eye(3)

cfg1 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.1, 0.1, 0.1]),
    initial_position=np.array([0.0, 0.0, 2.0]),
    initial_orientation=R0,
    size=(0.6, 0.2, 0.2),
    color=(1.0, 0.2, 0.2, 1.0)
)

cfg2 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.1, 0.1, 0.1]),
    initial_position=np.array([1.0, 0.0, 3.0]),
    initial_orientation=R1,
    size=(0.6, 0.2, 0.2),
    color=(0.2, 1.0, 0.2, 1.0)
)

cfg3 = BodyConfig(
    mass=1.0,
    inertia_matrix=np.diag([0.1, 0.1, 0.1]),
    initial_position=np.array([2.0, 0.0, 4.0]),
    initial_orientation=R2,
    size=(0.6, 0.2, 0.2),
    color=(0.2, 0.2, 1.0, 1.0)
)

controller1 = FullPosePDController(
    desired_position=[2.0, 0.0, 2.0],
    desired_orientation=desired_R,
    kp_pos=4.0,
    kd_pos=4.0,
    kp_att=1.0,
    kd_att=1.0,
    mass=cfg1.mass,
    gravity=env.gravity,
    force_limit=30.0,
    torque_limit=0.5
)

controller2 = FullPosePDController(
    desired_position=[2.0, 1.0, 3.0],
    desired_orientation=desired_R,
    kp_pos=4.0,
    kd_pos=4.0,
    kp_att=1.0,
    kd_att=1.0,
    mass=cfg2.mass,
    gravity=env.gravity,
    force_limit=30.0,
    torque_limit=0.5
)

controller3 = FullPosePDController(
    desired_position=[2.0, -1.0, 4.0],
    desired_orientation=desired_R,
    kp_pos=4.0,
    kd_pos=4.0,
    kp_att=1.0,
    kd_att=1.0,
    mass=cfg3.mass,
    gravity=env.gravity,
    force_limit=30.0,
    torque_limit=0.5
)

agents = [
    Agent(agent_id=0, config=cfg1, controller=controller1),
    Agent(agent_id=1, config=cfg2, controller=controller2),
    Agent(agent_id=2, config=cfg3, controller=controller3)
]

sim = MultiAgentSimulator(
    agents=agents,
    environment=env,
    time_step=0.001
)



In [ ]:
sim.simulate(duration=5.0, realtime_rate=1.0)

states = sim.get_all_agent_states()

for agent_id, state in states.items():
    print("Agent:", agent_id)
    print("Final position:", state.position)
    print("Final velocity:", state.linear_velocity)

In [ ]:
sim.simulate(duration=5.0, realtime_rate=1.0)

states = sim.get_all_agent_states()

for agent_id, state in states.items():
    print("Agent:", agent_id)
    print("Final position:", state.position)
    print("Final linear velocity:", state.linear_velocity)
    print("Final orientation:")
    print(state.orientation)
    print("Final angular velocity:", state.angular_velocity)

In [ ]:
#For attitude controller test
def rotation_error_angle(R_des, R):
    """
    Return orientation error angle in radians.
    """
    R_err = R_des.T @ R

    cos_angle = (np.trace(R_err) - 1.0) / 2.0
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    return np.arccos(cos_angle)


In [ ]:
for agent_id, state in states.items():
    angle_error = rotation_error_angle(desired_R, state.orientation)

    print("Agent:", agent_id)
    print("Final position:", state.position)
    print("Final position error:", controller1.desired_position - state.position)
    print("Final orientation error in degrees:", np.degrees(angle_error))
    print("Final linear velocity:", state.linear_velocity)
    print("Final angular velocity:", state.angular_velocity)